In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [3]:
# Load metadata
metadata_path = 'HAM10000_metadata.csv'
metadata = pd.read_csv(metadata_path)
metadata.head()

,lesion_id,image_id,dx,dx_type,age,sex,localization,dataset
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp,vidir_modern
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp,vidir_modern
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp,vidir_modern
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp,vidir_modern
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear,vidir_modern


In [14]:
# Paths
images_folder = 'images'

In [15]:
# Preprocess the labels
label_encoder = LabelEncoder()
metadata['label'] = label_encoder.fit_transform(metadata['dx'])

In [16]:
# Load images and labels
image_size = (128, 128)  # Resize images to 128x128
images = []
labels = []

for idx, row in metadata.iterrows():
    image_path = os.path.join(images_folder, row['image_id'] + '.jpg')
    if os.path.exists(image_path):
        img = load_img(image_path, target_size=image_size)
        img_array = img_to_array(img) / 255.0  # Normalize pixel values
        images.append(img_array)
        labels.append(row['label'])

images = np.array(images)
labels = np.array(labels)

In [17]:
# Split data into training and validation sets
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    images, labels, 
    test_size=0.2, 
    random_state=42
)


In [18]:
# Data augmentation
train_datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True
)
val_datagen = ImageDataGenerator()

train_generator = train_datagen.flow(X_train, y_train, batch_size=32)
val_generator = val_datagen.flow(X_val, y_val, batch_size=32)

In [19]:
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(128, 128, 3)),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(len(np.unique(labels)), activation='softmax')
])


c:\coding\web devlopment\miniproject\SkinAI-A_Dermatology_Assistant\.venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [20]:
# Compile the model
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

In [21]:
# Train the model
epochs = 100
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=epochs
)

c:\coding\web devlopment\miniproject\SkinAI-A_Dermatology_Assistant\.venv\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 31s 233ms/step - accuracy: 0.6900 - loss: 1.0638 - val_accuracy: 0.6620 - val_loss: 0.9935
Epoch 2/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 26s 206ms/step - accuracy: 0.6917 - loss: 0.9699 - val_accuracy: 0.6620 - val_loss: 0.9526
Epoch 3/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 37s 297ms/step - accuracy: 0.6927 - loss: 0.9415 - val_accuracy: 0.6630 - val_loss: 0.9128
Epoch 4/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 37s 294ms/step - accuracy: 0.6930 - loss: 0.8969 - val_accuracy: 0.6640 - val_loss: 0.8878
Epoch 5/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 41s 326ms/step - accuracy: 0.6967 - loss: 0.8717 - val_accuracy: 0.6710 - val_loss: 0.8504
Epoch 6/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 36s 290ms/step - accuracy: 0.6908 - loss: 0.8548 - val_accuracy: 0.6630 - val_loss: 0.8817
Epoch 7/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 37s 297ms/step - accuracy: 0.6915 - loss: 0.8466 - val_accuracy: 0.6740 - val_loss: 0.8501
Epoch 8/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 35s 283ms/step - accuracy: 0.7000 -

In [ ]:
# Save the model
model.save('SkinModel.h5')
print("Model training complete. 'SkinModel.h5' saved.")

Model training complete. 'SkinModel.h5' saved.
